# 00 — Data Transform

Loads `data/master_data.csv` (16 categories, 4,800 raw rows), cleans it
(dedup, drop degenerate rows), splits into a stratified train/test set,
creates the 5%-per-class semi-supervised labeled/unlabeled split, and
attaches a pre-generated summary sentence per row from
`data/master_data_summarized.csv` — feeds the unsupervised track's
summary-embedding clustering in `01`-`03`; carried along on every row for
every other method's full-output CSV too, though they train on raw `text`,
not `summary`. Run this first — every other notebook reads from
`data/processed/`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import pandas as pd

from utils import config
from utils.data import load_master_data, stratified_train_test_split, make_splits, stratified_sample

In [2]:
master_df = load_master_data(config.MASTER_DATA_PATH, config.CLASS_NAMES, min_words=config.MIN_WORDS)

assert master_df["text"].isna().sum() == 0
assert master_df["text"].duplicated().sum() == 0
assert set(master_df["category"].unique()) == set(config.CLASS_NAMES)

class_counts = master_df["label"].value_counts().sort_index()
print(f"Loaded {len(master_df)} cleaned rows (from 4,800 raw) across {config.NUM_CLASSES} classes")
print(class_counts)
assert (class_counts >= 250).all(), f"unexpectedly large class-size drop after cleaning:\n{class_counts}"

Loaded 4781 cleaned rows (from 4,800 raw) across 16 classes
label
0     300
1     300
2     300
3     299
4     300
5     297
6     292
7     300
8     299
9     300
10    299
11    300
12    300
13    300
14    295
15    300
Name: count, dtype: int64


In [3]:
master_df = stratified_sample(master_df, config.MASTER_SAMPLE_SIZE, seed=config.SEED, label_col="label")
print(f"Capped to {len(master_df)} rows (MASTER_SAMPLE_SIZE={config.MASTER_SAMPLE_SIZE})")
print(master_df["label"].value_counts().sort_index())

Capped to 4781 rows (MASTER_SAMPLE_SIZE=None)
label
0     300
1     300
2     300
3     299
4     300
5     297
6     292
7     300
8     299
9     300
10    299
11    300
12    300
13    300
14    295
15    300
Name: count, dtype: int64


In [4]:
train_clean, test_clean = stratified_train_test_split(
    master_df, test_fraction=config.TEST_FRACTION, seed=config.SEED)

assert len(train_clean) + len(test_clean) == len(master_df)
print(f"Train: {len(train_clean)} | Test: {len(test_clean)}")
print("Train class distribution:\n", train_clean["label"].value_counts().sort_index())
print("\nTest class distribution:\n", test_clean["label"].value_counts().sort_index())

Train: 3824 | Test: 957
Train class distribution:
 label
0     240
1     240
2     240
3     239
4     240
5     237
6     234
7     240
8     239
9     240
10    239
11    240
12    240
13    240
14    236
15    240
Name: count, dtype: int64

Test class distribution:
 label
0     60
1     60
2     60
3     60
4     60
5     60
6     58
7     60
8     60
9     60
10    60
11    60
12    60
13    60
14    59
15    60
Name: count, dtype: int64


In [5]:
# Load pre-generated summaries from master_data_summarized.csv and join
# them onto train_clean / test_clean by exact `text` match.
# IMPORTANT: load_master_data combines title+text into `text`, so we must
# apply the same merge to the summarized CSV before building the lookup dict.
summarized_raw = pd.read_csv(config.SUMMARIZED_DATA_PATH)
summarized_raw["text_merged"] = (
    summarized_raw["title"].fillna("") + " " + summarized_raw["text"].fillna("")
).str.strip()
text_to_summary = (
    summarized_raw.dropna(subset=["summary"])
    .set_index("text_merged")["summary"]
    .to_dict()
)

train_clean = train_clean.copy()
test_clean = test_clean.copy()

train_clean["summary"] = train_clean["text"].map(text_to_summary)
test_clean["summary"] = test_clean["text"].map(text_to_summary)

n_missing_train = int(train_clean["summary"].isna().sum())
n_missing_test = int(test_clean["summary"].isna().sum())
if n_missing_train or n_missing_test:
    print(f"WARNING: {n_missing_train} train / {n_missing_test} test rows have no summary match")
else:
    print("All summaries joined successfully.")

assert train_clean["summary"].isna().sum() == 0, "train_clean has rows without a summary"
assert test_clean["summary"].isna().sum() == 0, "test_clean has rows without a summary"
print(f"Train: {len(train_clean)} | Test: {len(test_clean)}")
print("Example summary:\n ", train_clean["summary"].iloc[0])

All summaries joined successfully.
Train: 3824 | Test: 957
Example summary:
  Well, this is different: Thousands of people are tuning in to a live broadcast of a virtual deer walking around the world of "Grand Theft Auto V." It scales mountains, visits ATMs, gets shot by police -- you know, the usual deer stuff. The video stream is hosted on Twitch, the game-broadcasting website Amazon acquired in 2014 for nearly $1 billion. It began last year as an art project and is still going strong.


In [6]:
labeled_df, unlabeled_df = make_splits(
    train_clean, label_fraction=config.LABEL_FRACTION, seed=config.SEED)

assert len(labeled_df) + len(unlabeled_df) == len(train_clean)
assert (unlabeled_df["label"] == -1).all()
print(f"Labeled pool: {len(labeled_df)} rows ({config.LABEL_FRACTION:.0%})")
print(f"Unlabeled pool: {len(unlabeled_df)} rows (true_label hidden for eval only)")
print("Labeled seed per-class counts:\n", labeled_df["label"].value_counts().sort_index())

Labeled pool: 192 rows (5%)
Unlabeled pool: 3632 rows (true_label hidden for eval only)
Labeled seed per-class counts:
 label
0     12
1     12
2     12
3     12
4     12
5     12
6     12
7     12
8     12
9     12
10    12
11    12
12    12
13    12
14    12
15    12
Name: count, dtype: int64


In [7]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_clean.to_parquet(config.PROCESSED_DIR / "train_clean.parquet", index=False)
test_clean.to_parquet(config.PROCESSED_DIR / "test_clean.parquet", index=False)
labeled_df.to_parquet(config.PROCESSED_DIR / "labeled.parquet", index=False)
unlabeled_df.to_parquet(config.PROCESSED_DIR / "unlabeled.parquet", index=False)
print("Saved processed splits (with generated summaries) to", config.PROCESSED_DIR)

Saved processed splits (with generated summaries) to C:\Users\ACER\OneDrive\Documents\final-project\data\processed
